In [ ]:
# from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
import os
import csv
import time
import torch
from transformers import DataCollatorForLanguageModeling
from transformers import AutoTokenizer
from transformers import GPT2Config, GPT2LMHeadModel
from transformers import TrainingArguments, Trainer
from transformers import TrainerCallback

In [2]:
# pip install transformers torch

In [3]:
# pip -q install "accelerate>=0.27.0"

In [ ]:
import transformers, accelerate
print(transformers.__version__)
print(accelerate.__version__)

In [4]:
# HF_TOKEN = userdata.get('HF_TOKEN')
from huggingface_hub import login
login()

In [5]:
dataset = load_dataset("azizdevlab/uzbek_corpus")

In [25]:
tokenizer = AutoTokenizer.from_pretrained("azizdevlab/gpt2-small-uzbek")

In [ ]:
print(tokenizer)
special_tokens_dict = {
    "pad_token": "<PAD>",
    "bos_token": "<BOS>",
    "eos_token": "<EOS>",
}

# Добавляем токены к токенизатору
tokenizer.add_special_tokens(special_tokens_dict)
print(tokenizer)
print("PAD token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("BOS token:", tokenizer.bos_token, tokenizer.bos_token_id)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)

0

In [ ]:
print(dataset)
dataset = dataset['train'].train_test_split(test_size=0.05)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 692471
    })
})

In [44]:
config = GPT2Config(
    vocab_size=tokenizer.vocab_size, #20000
    n_positions=768,
    n_ctx=768,
    n_embd=704,
    n_layer=12,
    n_head=11,
    n_inner=2816,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)
model =GPT2LMHeadModel(config)

In [47]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
log_file = "./loss_log.csv"
log_file = "./loss_log.csv"

if not os.path.exists(log_file):
    with open(log_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "step",
            "epoch",
            "lr",
            "train_loss",
            "eval_loss",
            "wall_time"
        ])

In [ ]:
class LogLossCallback(TrainerCallback):
    def __init__(self, log_file):
        self.log_file = log_file
        self.last_train_loss = None
        self.last_eval_loss = None
        self.last_lr = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        # если DDP — пишем только из главного процесса
        if not state.is_world_process_zero:
            return

        if "loss" in logs:
            self.last_train_loss = logs["loss"]

        if "eval_loss" in logs:
            self.last_eval_loss = logs["eval_loss"]

        if "learning_rate" in logs:
            self.last_lr = logs["learning_rate"]

        with open(self.log_file, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                state.global_step,
                state.epoch,
                self.last_lr,
                self.last_train_loss,
                self.last_eval_loss,
                time.time()
            ])

In [ ]:
training_args = TrainingArguments(
    output_dir="./model_out",
    # training
    num_train_epochs=4,
    per_device_train_batch_size=16,#32
    per_device_eval_batch_size=16, #32
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=160,

    # Evaluation / Logging
    eval_strategy="steps",
    eval_steps=25,
    logging_strategy="steps",
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    save_strategy="epoch",
    report_to="none",  # важно, чтобы не было конфликтов
)

In [74]:
trainer = Trainer(model=model,
                 args = training_args,
                 train_dataset=dataset["train"],
                 eval_dataset=dataset["test"],
                 data_collator = data_collator,
                 callbacks=[LogLossCallback(log_file)]
                )

In [ ]:
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Всего памяти: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
    print(f"  Используется: {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
    print(f"  Зарезервировано: {torch.cuda.memory_reserved(i) / 1e9:.2f} GB\n")

In [75]:
trainer.train()

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
os.getcwd()